# 2단계 피처 엔지니어링

1단계에서 생성한 도시별 샘플 CSV를 Google Drive에서 불러오고, 텍스트/날짜/유저/식당/상권 클러스터 파생변수를 추가한 뒤 `*_features.csv` 파일로 저장한다.

이 노트북은 Colab 실행을 기준으로 작성되었다.

## 1. 환경 설정 및 경로 확인

통합 노트북에서는 앞 단계에서 이미 `PROJECT_ROOT`가 설정되어 있을 수 있다. 이 셀은 기존 경로가 유효하면 그대로 재사용하고, 없을 때만 실행 환경에 맞게 프로젝트 루트를 다시 찾는다.

- Colab에서는 Google Drive의 `Team-6/` 폴더를 우선 확인한다.
- 로컬에서는 현재 작업 디렉터리와 상위 폴더를 탐색해 `data/`, `notebooks/`가 있는 저장소 루트를 찾는다.

따라서 1단계나 2-1단계를 건너뛰고 이 셀부터 실행해도, 저장소에 포함된 `data/interim/` 파일을 기준으로 다음 단계가 이어지도록 구성한다.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ModuleNotFoundError:
        return False


IN_COLAB = running_in_colab()


def valid_project_root(path):
    path = Path(path).expanduser().resolve()
    return (path / 'data').exists() and (path / 'notebooks').exists()


def find_project_root():
    existing_root = globals().get('PROJECT_ROOT')
    if existing_root is not None and valid_project_root(existing_root):
        print(f'기존 프로젝트 루트 재사용: {Path(existing_root).resolve()}')
        return Path(existing_root).resolve()

    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
        colab_root = Path('/content/drive/MyDrive/ml_project/Team-6')
        if valid_project_root(colab_root):
            print(f'Colab 프로젝트 루트 확인: {colab_root}')
            return colab_root

    current_dir = Path.cwd().resolve()
    for candidate in [current_dir, *current_dir.parents]:
        if valid_project_root(candidate):
            print(f'로컬 프로젝트 루트 확인: {candidate}')
            return candidate

    raise FileNotFoundError(
        '프로젝트 루트를 찾을 수 없다. '
        'data/와 notebooks/ 폴더가 있는 Team-6 저장소 안에서 노트북을 실행한다.'
    )


PROJECT_ROOT = find_project_root()


def find_input_dir(project_root):
    candidates = [
        project_root / 'data' / 'interim',
    ]
    for candidate in candidates:
        if (candidate / 'yelp_subset_philly_15k.csv').exists():
            print(f'입력 폴더 확인: {candidate}')
            return candidate

    matches = list(project_root.rglob('yelp_subset_philly_15k.csv'))
    if matches:
        detected_dir = matches[0].parent
        print(f'입력 폴더 확인: {detected_dir}')
        return detected_dir

    raise FileNotFoundError(f'프로젝트 경로 아래에서 yelp_subset_philly_15k.csv를 찾을 수 없다: {project_root}')


INPUT_DIR = find_input_dir(PROJECT_ROOT)
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
TABLE_DIR = PROJECT_ROOT / 'output' / 'tables'
REPORT_DIR = PROJECT_ROOT / 'output' / 'reports'
for directory in [OUTPUT_DIR, TABLE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DRIVE_PROJECT_ROOT = PROJECT_ROOT.parent
BUSINESS_LOCATION_CSV_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'interim' / 'business_location.csv',
    PROJECT_ROOT / 'data' / 'business_location.csv',
]

BUSINESS_JSON_CANDIDATES = [
    Path('/content/yelp_data/yelp_academic_dataset_business.json'),
    PROJECT_ROOT / 'data' / 'raw' / 'yelp_academic_dataset_business.json',
    PROJECT_ROOT / 'data' / 'yelp_academic_dataset_business.json',
]

CITY_CONFIGS = [
    {
        'city': 'Philadelphia',
        'input_file': 'yelp_subset_philly_15k.csv',
        'output_file': 'yelp_subset_philly_15k_features.csv',
    },
    {
        'city': 'Tucson',
        'input_file': 'yelp_subset_tucson_15k.csv',
        'output_file': 'yelp_subset_tucson_15k_features.csv',
    },
    {
        'city': 'New Orleans',
        'input_file': 'yelp_subset_new_orleans_15k.csv',
        'output_file': 'yelp_subset_new_orleans_15k_features.csv',
    },
]

KMEANS_CLUSTERS = 5
RANDOM_STATE = 42

print(f'프로젝트 루트: {PROJECT_ROOT}')
print(f'입력 폴더: {INPUT_DIR}')
print(f'출력 폴더: {OUTPUT_DIR}')
print(f'표 저장 폴더: {TABLE_DIR}')
print(f'보고서 저장 폴더: {REPORT_DIR}')


## 2. 입력 파일 확인

In [ ]:
def check_input_files(city_configs=CITY_CONFIGS):
    missing_files = []
    for config in city_configs:
        input_path = INPUT_DIR / config['input_file']
        if not input_path.exists():
            missing_files.append(input_path)

    if missing_files:
        missing_text = '\n'.join(str(path) for path in missing_files)
        raise FileNotFoundError(f'입력 CSV 파일을 찾을 수 없다:\n{missing_text}')

    print('입력 CSV 파일 확인 완료')


check_input_files()

## 3. 공통 함수 정의

In [ ]:
def add_text_features(df):
    df = df.copy()
    df['text'] = df['text'].fillna('').astype(str)

    df['text_length'] = df['text'].str.len()
    df['word_count'] = df['text'].str.split().str.len()
    df['sentence_count'] = df['text'].str.count(r'[.!?]+').clip(lower=1)
    df['avg_word_length'] = df['text_length'] / df['word_count'].clip(lower=1)
    df['uppercase_ratio'] = df['text'].apply(
        lambda text: sum(1 for char in text if char.isupper()) / max(len(text), 1)
    )
    df['exclamation_count'] = df['text'].str.count('!')
    df['question_count'] = df['text'].str.count(r'\?')
    df['engagement_sum'] = df[['useful', 'funny', 'cool']].fillna(0).sum(axis=1)
    return df


def add_date_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    df['review_year'] = df['date'].dt.year
    df['review_month'] = df['date'].dt.month
    df['review_dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['review_dayofweek'].isin([5, 6]).astype(int)
    return df


def add_user_features(df):
    df = df.copy()
    user_stats = df.groupby('user_id')['stars'].agg(
        user_review_count='count',
        user_avg_stars='mean'
    ).reset_index()

    df = df.merge(user_stats, on='user_id', how='left')
    df['user_star_deviation'] = df['stars'] - df['user_avg_stars']
    return df


def add_business_features(df):
    df = df.copy()
    business_stats = df.groupby('business_id')['stars'].agg(
        business_review_count='count',
        business_avg_stars='mean'
    ).reset_index()

    df = df.merge(business_stats, on='business_id', how='left')
    df['business_star_deviation'] = df['stars'] - df['business_avg_stars']
    return df


def find_business_location_csv_path():
    for path in BUSINESS_LOCATION_CSV_CANDIDATES:
        if path.exists():
            print(f'사업장 위치 CSV 확인: {path}')
            return path

    matches = list(DRIVE_PROJECT_ROOT.rglob('business_location.csv'))
    if matches:
        detected_path = matches[0]
        print(f'사업장 위치 CSV 확인: {detected_path}')
        return detected_path

    return None


def find_business_json_path():
    for path in BUSINESS_JSON_CANDIDATES:
        if path.exists():
            print(f'사업장 JSON 확인: {path}')
            return path

    matches = list(DRIVE_PROJECT_ROOT.rglob('yelp_academic_dataset_business.json'))
    if matches:
        detected_path = matches[0]
        print(f'사업장 JSON 확인: {detected_path}')
        return detected_path

    return None


def load_business_locations(business_ids):
    location_csv_path = find_business_location_csv_path()
    if location_csv_path is not None:
        location_df = pd.read_csv(location_csv_path)
        return location_df.loc[
            location_df['business_id'].isin(business_ids),
            ['business_id', 'latitude', 'longitude']
        ].drop_duplicates('business_id')

    business_json_path = find_business_json_path()
    if business_json_path is None:
        print('사업장 위치 CSV 또는 business.json을 찾지 못해 위치 기반 피처 생성을 건너뛴다.')
        print('필요 파일명: business_location.csv 또는 yelp_academic_dataset_business.json')
        print(f'검색한 경로: {DRIVE_PROJECT_ROOT}')
        return None

    print(f'사업장 위치 정보 불러오기: {business_json_path}')
    biz_df = pd.read_json(business_json_path, lines=True)
    location_df = biz_df.loc[
        biz_df['business_id'].isin(business_ids),
        ['business_id', 'latitude', 'longitude']
    ].drop_duplicates('business_id')
    return location_df


def add_location_features(df, location_df):
    df = df.copy()
    if location_df is None:
        df['latitude'] = np.nan
        df['longitude'] = np.nan
        return df

    return df.merge(location_df, on='business_id', how='left')


def add_kmeans_cluster(df, n_clusters=KMEANS_CLUSTERS, random_state=RANDOM_STATE):
    df = df.copy()
    df['business_cluster'] = np.nan

    coords = df[['latitude', 'longitude']].dropna()
    if coords.empty:
        print('사용 가능한 좌표가 없어 K-Means 클러스터링을 건너뛴다.')
        return df

    unique_count = coords.drop_duplicates().shape[0]
    cluster_count = min(n_clusters, unique_count)
    if cluster_count < 2:
        print('고유 좌표 수가 부족해 K-Means 클러스터링을 건너뛴다.')
        return df

    scaler = StandardScaler()
    coords_scaled = scaler.fit_transform(coords)

    kmeans = KMeans(n_clusters=cluster_count, random_state=random_state, n_init='auto')
    df.loc[coords.index, 'business_cluster'] = kmeans.fit_predict(coords_scaled)
    df['business_cluster'] = df['business_cluster'].astype('Int64')
    return df


def add_all_features(df, location_df):
    df = add_text_features(df)
    df = add_date_features(df)
    df = add_user_features(df)
    df = add_business_features(df)
    df = add_location_features(df, location_df)
    df = add_kmeans_cluster(df)
    return df


def summarize_features(df):
    print('데이터 크기:', df.shape)
    print('\n타깃 분포:')
    print(df['is_positive'].value_counts(dropna=False))
    print('\n결측치가 많은 컬럼:')
    print(df.isnull().sum().sort_values(ascending=False).head(15))

## 4. 위치 정보 로드

`business.json`이 `/content/yelp_data` 또는 Drive의 `Team-6/data`에 있으면 위도/경도를 병합하고 K-Means 클러스터를 생성한다. 파일이 없으면 위치 기반 피처만 건너뛴다.

In [ ]:
all_business_ids = set()
for config in CITY_CONFIGS:
    temp_df = pd.read_csv(INPUT_DIR / config['input_file'], usecols=['business_id'])
    all_business_ids.update(temp_df['business_id'].dropna().unique())

location_df = load_business_locations(all_business_ids)
if location_df is not None:
    print('위치 정보 행 수:', location_df.shape[0])
    display(location_df.head())

## 5. 도시별 피처 생성 및 저장

In [ ]:
feature_datasets = {}

for config in CITY_CONFIGS:
    city = config['city']
    input_path = INPUT_DIR / config['input_file']
    output_path = OUTPUT_DIR / config['output_file']

    print(f'\n[{city}] 피처 생성 시작')
    df = pd.read_csv(input_path)
    print('원본 데이터 크기:', df.shape)

    feature_df = add_all_features(df, location_df)
    summarize_features(feature_df)

    feature_df.to_csv(output_path, index=False)
    feature_datasets[city] = feature_df
    print(f'저장 완료: {output_path}')

print('\n전체 피처 생성 완료')

## 6. 결과 확인

In [ ]:
for config in CITY_CONFIGS:
    output_path = OUTPUT_DIR / config['output_file']
    print(output_path, output_path.exists())

sample_city = CITY_CONFIGS[0]['city']
display(feature_datasets[sample_city].head())

## 7. 최종 CSV 저장 및 산출물 인덱스

최종 feature CSV를 `data/processed/`에 저장하고, 도시별 행 수/컬럼 수/파일 크기를 `output/tables/feature_engineering_outputs.csv`로 남긴다.


In [ ]:
saved_outputs = []

for config in CITY_CONFIGS:
    city = config['city']
    output_path = OUTPUT_DIR / config['output_file']

    if city not in feature_datasets:
        raise KeyError(f"feature_datasets에서 {city} 데이터를 찾을 수 없다. 피처 생성 셀을 먼저 실행한다.")

    feature_datasets[city].to_csv(output_path, index=False)
    saved_outputs.append({
        'city': city,
        'path': str(output_path),
        'rows': len(feature_datasets[city]),
        'columns': feature_datasets[city].shape[1],
        'file_size_mb': output_path.stat().st_size / 1024 / 1024,
    })

saved_outputs_df = pd.DataFrame(saved_outputs)
summary_path = TABLE_DIR / 'feature_engineering_outputs.csv'
saved_outputs_df.to_csv(summary_path, index=False)
display(saved_outputs_df)
print(f'최종 피처 CSV 저장 완료. 요약 파일: {summary_path}')
